In [2]:
import pandas as pd

In [6]:
try:
    gateway_df = pd.read_csv('/content/gateway.csv')
    print("First 5 rows of gateway.csv:")
    display(gateway_df.head())

    # Check for unique transaction IDs
    if 'transaction_id' in gateway_df.columns:
        if gateway_df['transaction_id'].is_unique:
            print("All 'transaction_id' in gateway.csv are unique.")
        else:
            print("Some 'transaction_id' in gateway.csv are not unique.")
            duplicate_transactions = gateway_df[gateway_df.duplicated(subset=['transaction_id'], keep=False)]
            print("Duplicate 'transaction_id' records in gateway.csv:")
            display(duplicate_transactions.sort_values(by='transaction_id'))
    else:
        print("Column 'transaction_id' not found in gateway.csv. Please specify the correct column name for transaction IDs.")

    print("\nNull values in gateway.csv:")
    display(gateway_df.isnull().sum())

except FileNotFoundError:
    print("Error: gateway.csv not found. Please ensure the file is uploaded to '/content/'.")
except Exception as e:
    print(f"An error occurred while processing gateway.csv: {e}")

First 5 rows of gateway.csv:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


All 'transaction_id' in gateway.csv are unique.

Null values in gateway.csv:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0


In [8]:
try:
    ledger_df = pd.read_csv('/content/ledger.csv')
    print("First 5 rows of ledger.csv:")
    display(ledger_df.head())

    # Check for unique transaction IDs
    if 'transaction_id' in ledger_df.columns:
        if ledger_df['transaction_id'].is_unique:
            print("All 'transaction_id' in ledger.csv are unique.")
        else:
            print("Some 'transaction_id' in ledger.csv are not unique.")
            duplicate_transactions = ledger_df[ledger_df.duplicated(subset=['transaction_id'], keep=False)]
            print("Duplicate 'transaction_id' records in ledger.csv:")
            display(duplicate_transactions.sort_values(by='transaction_id'))
    else:
        print("Column 'transaction_id' not found in ledger.csv. Please specify the correct column name for transaction IDs.")

    print("\nNull values in ledger.csv:")
    display(ledger_df.isnull().sum())

except FileNotFoundError:
    print("Error: ledger.csv not found. Please ensure the file is uploaded to '/content/'.")
except Exception as e:
    print(f"An error occurred while processing ledger.csv: {e}")

First 5 rows of ledger.csv:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


All 'transaction_id' in ledger.csv are unique.

Null values in ledger.csv:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0


## Identifying Records Missing in Gateway

In [9]:
# Merge ledger_df with gateway_df to find missing records
merged_df = ledger_df.merge(gateway_df, on='transaction_id', how='left', suffixes=('_ledger', '_gateway'))

# Identify records where 'transaction_id' from gateway_df is null (meaning it's in ledger but not gateway)
missing_in_gateway = merged_df[merged_df['transaction_date_gateway'].isnull()]

print("Records in ledger.csv but missing in gateway.csv:")
display(missing_in_gateway)

Records in ledger.csv but missing in gateway.csv:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
3,R004,2026-03-02,M003,2100.0,success,Card,NaN,NaN,NaN,NaN,NaN
9,R010,2026-03-05,M004,2500.0,success,Wallet,NaN,NaN,NaN,NaN,NaN


## Identifying Records Missing in Ledger

In [10]:
# Merge gateway_df with ledger_df to find missing records
merged_df_rev = gateway_df.merge(ledger_df, on='transaction_id', how='left', suffixes=('_gateway', '_ledger'))

# Identify records where 'transaction_id' from ledger_df is null (meaning it's in gateway but not ledger)
missing_in_ledger = merged_df_rev[merged_df_rev['transaction_date_ledger'].isnull()]

print("Records in gateway.csv but missing in ledger.csv:")
display(missing_in_ledger)

Records in gateway.csv but missing in ledger.csv:


,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
8,R011,2026-03-05,M003,1800.0,success,Card,NaN,NaN,NaN,NaN,NaN


## Consolidated Table of Missing Transactions

In [11]:
# Prepare records missing in gateway for consolidation
missing_in_gateway_consolidated = missing_in_gateway[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'amount_usd_ledger',
    'status_ledger',
    'payment_method_ledger'
]].copy()
missing_in_gateway_consolidated.columns = [
    'transaction_id',
    'transaction_date',
    'merchant_id',
    'amount_usd',
    'status',
    'payment_method'
]
missing_in_gateway_consolidated['missing_from'] = 'gateway'

# Prepare records missing in ledger for consolidation
missing_in_ledger_consolidated = missing_in_ledger[[
    'transaction_id',
    'transaction_date_gateway',
    'merchant_id_gateway',
    'amount_usd_gateway',
    'status_gateway',
    'payment_method_gateway'
]].copy()
missing_in_ledger_consolidated.columns = [
    'transaction_id',
    'transaction_date',
    'merchant_id',
    'amount_usd',
    'status',
    'payment_method'
]
missing_in_ledger_consolidated['missing_from'] = 'ledger'

# Consolidate both missing sets into a single table
consolidated_missing_transactions = pd.concat([
    missing_in_gateway_consolidated,
    missing_in_ledger_consolidated
], ignore_index=True)

print("Consolidated table of transactions missing from either gateway or ledger:")
display(consolidated_missing_transactions)

Consolidated table of transactions missing from either gateway or ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,missing_from
0,R004,2026-03-02,M003,2100.0,success,Card,gateway
1,R010,2026-03-05,M004,2500.0,success,Wallet,gateway
2,R011,2026-03-05,M003,1800.0,success,Card,ledger


## Identifying Amount Mismatches

In [12]:
# Merge both dataframes on transaction_id with an inner join to only get common transactions
mismatched_amounts_df = pd.merge(gateway_df, ledger_df, on='transaction_id', how='inner', suffixes=('_gateway', '_ledger'))

# Filter for rows where amount_usd is not equal and both values are not null
mismatched_amounts_df = mismatched_amounts_df[
    (mismatched_amounts_df['amount_usd_gateway'] != mismatched_amounts_df['amount_usd_ledger']) &
    (mismatched_amounts_df['amount_usd_gateway'].notna()) &
    (mismatched_amounts_df['amount_usd_ledger'].notna())
]

# Select relevant columns for display
mismatched_amounts_display = mismatched_amounts_df[[
    'transaction_id',
    'amount_usd_gateway',
    'amount_usd_ledger'
]]

print("Transactions with amount mismatches between gateway and ledger:")
display(mismatched_amounts_display)

Transactions with amount mismatches between gateway and ledger:


,transaction_id,amount_usd_gateway,amount_usd_ledger
1,R002,900.0,850.0
6,R008,600.0,640.0


In [14]:
mismatched_amounts_display = mismatched_amounts_display.copy()
mismatched_amounts_display['discrepancy'] = abs(mismatched_amounts_display['amount_usd_gateway'] - mismatched_amounts_display['amount_usd_ledger'])
total_discrepancy = mismatched_amounts_display['discrepancy'].sum()

print(f"Total discrepancy amount for mismatched records: {total_discrepancy:.2f}")

Total discrepancy amount for mismatched records: 90.00


## Identifying Status Mismatches

In [16]:
# Merge both dataframes on transaction_id with an inner join to only get common transactions
mismatched_status_df = pd.merge(gateway_df, ledger_df, on='transaction_id', how='inner', suffixes=('_gateway', '_ledger'))

# Filter for rows where status is not equal and both values are not null
mismatched_status_df = mismatched_status_df[
    (mismatched_status_df['status_gateway'] != mismatched_status_df['status_ledger']) &
    (mismatched_status_df['status_gateway'].notna()) &
    (mismatched_status_df['status_ledger'].notna())
]

# Display all relevant columns for transactions with status mismatches
print("Full details for transactions with status mismatches between gateway and ledger:")
display(mismatched_status_df)

Full details for transactions with status mismatches between gateway and ledger:


,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
3,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card


## Final Reconciliation Report

### 1. Consolidated Missing Transactions

In [17]:
print("Transactions found in one system but missing from the other:")
display(consolidated_missing_transactions)

Transactions found in one system but missing from the other:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,missing_from
0,R004,2026-03-02,M003,2100.0,success,Card,gateway
1,R010,2026-03-05,M004,2500.0,success,Wallet,gateway
2,R011,2026-03-05,M003,1800.0,success,Card,ledger


### 2. Amount Mismatches

In [18]:
print("Transactions with differing amount_usd values:")
display(mismatched_amounts_display)

Transactions with differing amount_usd values:


,transaction_id,amount_usd_gateway,amount_usd_ledger,discrepancy
1,R002,900.0,850.0,50.0
6,R008,600.0,640.0,40.0


### 3. Status Mismatches

In [19]:
print("Transactions with differing status values:")
display(mismatched_status_df)

Transactions with differing status values:


,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
3,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card


### Summary

This report highlights the key discrepancies between the gateway and ledger records. Each section provides details on transactions that are:

*   **Missing from either system:** Critical for ensuring all transactions are recorded.
*   **Having amount discrepancies:** Important for financial accuracy.
*   **Having status discrepancies:** Essential for correct transaction state management.

Further investigation into each of these categories would be necessary to reconcile these differences.

## Summary Metrics

In [20]:
# Calculate and display summary metrics

num_missing_in_gateway = len(missing_in_gateway)
num_missing_in_ledger = len(missing_in_ledger)
total_missing_transactions = len(consolidated_missing_transactions)
num_amount_mismatches = len(mismatched_amounts_display)
num_status_mismatches = len(mismatched_status_df)

print(f"Number of transactions missing in Gateway: {num_missing_in_gateway}")
print(f"Number of transactions missing in Ledger: {num_missing_in_ledger}")
print(f"Total unique transactions missing from either system: {total_missing_transactions}")
print(f"Number of transactions with amount discrepancies: {num_amount_mismatches}")
print(f"Total discrepancy amount for mismatched records: {total_discrepancy:.2f}")
print(f"Number of transactions with status discrepancies: {num_status_mismatches}")

Number of transactions missing in Gateway: 2
Number of transactions missing in Ledger: 1
Total unique transactions missing from either system: 3
Number of transactions with amount discrepancies: 2
Total discrepancy amount for mismatched records: 90.00
Number of transactions with status discrepancies: 1


## Exporting Results

In [27]:
import json

# Export consolidated missing transactions to CSV
consolidated_missing_transactions.to_csv('consolidated_missing_transactions.csv', index=False)
print("Exported 'consolidated_missing_transactions.csv'")

# Export amount mismatches to CSV
mismatched_amounts_display.to_csv('amount_mismatches', index=False)
print("Exported 'amount_mismatches.csv'")

# Export status mismatches to CSV
mismatched_status_df.to_csv('mismatched_status.csv', index=False)
print("Exported 'mismatched_status.csv'")

# Calculate additional metrics for JSON export
total_ledger_rows = len(ledger_df)
total_gateway_rows = len(gateway_df)
reconciliation_issue_count = total_missing_transactions + num_amount_mismatches + num_status_mismatches
ledger_total_amount = ledger_df['amount_usd'].sum()
gateway_total_amount = gateway_df['amount_usd'].sum()
amount_at_risk = total_discrepancy

# Prepare summary metrics for JSON export
summary_metrics = {
    "total_ledger_rows": total_ledger_rows,
    "total_gateway_rows": total_gateway_rows,
    "missing_in_gateway_count": num_missing_in_gateway,
    "missing_in_ledger_count": num_missing_in_ledger,
    "amount_mismatch_count": num_amount_mismatches,
    "status_mismatch_count": num_status_mismatches,
    "reconciliation_issue_count": reconciliation_issue_count,
    "ledger_total_amount": ledger_total_amount,
    "gateway_total_amount": gateway_total_amount,
    "amount_at_risk": amount_at_risk
}

# Export summary metrics to JSON
with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)
print("Exported 'summary_metrics.json'")

Exported 'consolidated_missing_transactions.csv'
Exported 'amount_mismatches.csv'
Exported 'mismatched_status.csv'
Exported 'summary_metrics.json'


In [24]:
for key, value in summary_metrics.items():
    print(f"{key}: {value}")

total_ledger_rows: 10
total_gateway_rows: 9
missing_in_gateway_count: 2
missing_in_ledger_count: 1
amount_mismatch_count: 2
status_mismatch_count: 1
reconciliation_issue_count: 6
ledger_total_amount: 23340.0
gateway_total_amount: 20550.0
amount_at_risk: 90.0


## Examining `api_response_sample.json`

In [29]:
import json

try:
    with open('/content/api_response_sample.json', 'r') as f:
        api_response_data = json.load(f)
    print("Content of api_response_sample.json:")
    display(api_response_data)
except FileNotFoundError:
    print("Error: api_response_sample.json not found. Please ensure the file is uploaded to '/content/'.")
except json.JSONDecodeError:
    print("Error: Could not decode JSON from api_response_sample.json. Please check if it's a valid JSON file.")
except Exception as e:
    print(f"An error occurred: {e}")

Content of api_response_sample.json:


{'generated_at': '2026-03-07T10:00:00Z',
 'source': 'QuickPay Settlement API',
 'batches': [{'batch_id': 'B001',
   'merchant': {'merchant_id': 'M001',
    'merchant_name': 'Alpha Mart',
    'region': 'APAC'},
   'settlements': [{'settlement_id': 'S001',
     'amount_usd': 1520.5,
     'status': 'settled',
     'processed_at': '2026-03-07T08:10:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S002',
     'amount_usd': 980.0,
     'status': 'pending',
     'processed_at': '2026-03-07T08:45:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S003',
     'amount_usd': 640.0,
     'status': 'settled',
     'processed_at': '2026-03-07T09:15:00Z',
     'bank': {'name': 'Bank B', 'country': 'SG'}}]},
  {'batch_id': 'B002',
   'merchant': {'merchant_id': 'M004',
    'merchant_name': 'Delta Travels',
    'region': 'US'},
   'settlements': [{'settlement_id': 'S004',
     'amount_usd': 2100.0,
     'status': 'settled',
     'processed_at'

In [30]:
# Initialize an empty list to store the flattened records
flattened_data = []

# Iterate through each batch in the API response data
for batch in api_response_data['batches']:
    batch_id = batch['batch_id']
    merchant_id = batch['merchant']['merchant_id']
    merchant_name = batch['merchant']['merchant_name']
    merchant_region = batch['merchant']['region']

    # Iterate through each settlement within the current batch
    for settlement in batch['settlements']:
        flattened_record = {
            'batch_id': batch_id,
            'merchant_id': merchant_id,
            'merchant_name': merchant_name,
            'merchant_region': merchant_region,
            'settlement_id': settlement['settlement_id'],
            'amount_usd': settlement['amount_usd'],
            'status': settlement['status'],
            'processed_at': settlement['processed_at'],
            'bank_name': settlement['bank']['name'],
            'bank_country': settlement['bank']['country']
        }
        flattened_data.append(flattened_record)

# Convert the list of dictionaries to a pandas DataFrame
api_response_df = pd.DataFrame(flattened_data)

print("Flattened API Response DataFrame (first 5 rows):")
display(api_response_df.head())

Flattened API Response DataFrame (first 5 rows):


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US


In [31]:
# Function to convert column names to snake_case
def to_snake_case(name):
    return name.lower().replace(' ', '_')

# Apply the function to all column names in api_response_df
api_response_df.columns = [to_snake_case(col) for col in api_response_df.columns]

print("DataFrame with column names converted to snake_case (first 5 rows):")
display(api_response_df.head())

DataFrame with column names converted to snake_case (first 5 rows):


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US


In [32]:
api_response_df.rename(columns={'processed_at': 'Date & Time'}, inplace=True)

print("DataFrame with 'processed_at' column renamed to 'Date & Time' (first 5 rows):")
display(api_response_df.head())

DataFrame with 'processed_at' column renamed to 'Date & Time' (first 5 rows):


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,Date & Time,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US


In [33]:
api_response_df['Date & Time'] = pd.to_datetime(api_response_df['Date & Time'])

print("DataFrame with 'Date & Time' column converted to datetime (first 5 rows):")
display(api_response_df.head())

DataFrame with 'Date & Time' column converted to datetime (first 5 rows):


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,Date & Time,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US


In [34]:
api_response_df['Date & Time'] = api_response_df['Date & Time'].dt.tz_localize(None)

print("DataFrame with timezone information removed from 'Date & Time' (first 5 rows):")
display(api_response_df.head())

DataFrame with timezone information removed from 'Date & Time' (first 5 rows):


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,Date & Time,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07 08:10:00,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07 08:45:00,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07 09:15:00,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07 08:20:00,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07 08:50:00,Bank C,US


In [37]:
api_response_df.to_csv('api_normalized.csv', index=False)
print("Exported 'api_normalized.csv'")

Exported 'api_normalized.csv'


## Examining `cleaned_transactions.csv`

In [39]:
try:
    cleaned_transactions_df = pd.read_csv('/content/cleaned_transactions.csv')
    print("First 5 rows of cleaned_transactions.csv:")
    display(cleaned_transactions_df.head())
except FileNotFoundError:
    print("Error: cleaned_transactions.csv not found. Please ensure the file is uploaded to '/content/'.")
except Exception as e:
    print(f"An error occurred while processing cleaned_transactions.csv: {e}")

First 5 rows of cleaned_transactions.csv:


,transaction_id,transaction_date,merchant_name_clean,amount_usd,status_clean,risk_score_clean,gateway_region_clean,user_id,User Name,payment_method,...,.2,.3,.4,.5,.6,.7,.8,.9,.10,.11
0,T001,2026-03-01,Alpha Mart,4998.0,Captured,62,APAC,U001,Aarav Shah,UPI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T002,2026-03-01,Alpha Mart,2499.0,Captured,55,MISSING,U002,Meera Iyer,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T003,2026-03-01,Beta Stores,6069.0,Captured,71,APAC,U003,Kabir Jain,NetBanking,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T023,2026-03-01,City Pharma,5616.0,Captured,42,EU,U003,Kabir Jain,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T027,2026-03-01,Delta Travels,7200.0,Captured,49,US,U007,Noah Kim,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning `cleaned_transactions_df`

In [41]:
# Identify columns that start with '.'
columns_to_drop = [col for col in cleaned_transactions_df.columns if col.startswith('.')]

# Drop these columns from the DataFrame
if columns_to_drop:
    cleaned_transactions_df = cleaned_transactions_df.drop(columns=columns_to_drop)
    print(f"Dropped the following columns: {columns_to_drop}")
else:
    print("No columns starting with '.' to drop.")

print("Cleaned DataFrame (first 5 rows) after dropping unnamed columns:")
display(cleaned_transactions_df.head())

No columns starting with '.' to drop.
Cleaned DataFrame (first 5 rows) after dropping unnamed columns:


,transaction_id,transaction_date,merchant_name_clean,amount_usd,status_clean,risk_score_clean,gateway_region_clean,user_id,User Name,payment_method,...,.2,.3,.4,.5,.6,.7,.8,.9,.10,.11
0,T001,2026-03-01,Alpha Mart,4998.0,Captured,62,APAC,U001,Aarav Shah,UPI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T002,2026-03-01,Alpha Mart,2499.0,Captured,55,MISSING,U002,Meera Iyer,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T003,2026-03-01,Beta Stores,6069.0,Captured,71,APAC,U003,Kabir Jain,NetBanking,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T023,2026-03-01,City Pharma,5616.0,Captured,42,EU,U003,Kabir Jain,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T027,2026-03-01,Delta Travels,7200.0,Captured,49,US,U007,Noah Kim,Card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Daily Transaction Summary

In [42]:
# Ensure 'transaction_date' is in datetime format
cleaned_transactions_df['transaction_date'] = pd.to_datetime(cleaned_transactions_df['transaction_date'])

# Create the daily summary
daily_summary = cleaned_transactions_df.groupby('transaction_date').agg(
    transaction_count=('transaction_id', 'count'),
    total_amount=('amount_usd', 'sum')
).reset_index()

# Rename columns for clarity
daily_summary.rename(columns={'transaction_date': 'Date', 'transaction_count': 'Transaction Count', 'total_amount': 'Total Amount'}, inplace=True)

print("Daily Transaction Summary:")
display(daily_summary)

Daily Transaction Summary:


,Date,Transaction Count,Total Amount
0,2026-03-01,5,26382.0
1,2026-03-02,6,25049.0
2,2026-03-03,5,18391.0
3,2026-03-04,5,16420.0
4,2026-03-05,6,19232.0
5,2026-03-06,3,10606.0


In [43]:
daily_summary.to_csv('daily_summary.csv', index=False)
print("Exported 'daily_summary.csv'")

Exported 'daily_summary.csv'


### Payment Method Summary

In [44]:
# Group by payment_method and aggregate transaction count and total amount
payment_method_summary = cleaned_transactions_df.groupby('payment_method').agg(
    transaction_count=('transaction_id', 'count'),
    total_amount=('amount_usd', 'sum')
).reset_index()

# Rename columns for clarity
payment_method_summary.rename(columns={'transaction_count': 'Transaction Count', 'total_amount': 'Total Amount'}, inplace=True)

print("Payment Method Summary:")
display(payment_method_summary)

Payment Method Summary:


,payment_method,Transaction Count,Total Amount
0,Card,13,52972.0
1,NetBanking,3,15306.0
2,UPI,9,34397.5
3,Wallet,5,13404.5


In [45]:
payment_method_summary.to_csv('payment_method_breakdown.csv', index=False)
print("Exported 'payment_method_breakdown.csv'")

Exported 'payment_method_breakdown.csv'


### Region Breakdown from Cleaned Transactions

In [51]:
# Group by 'gateway_region_clean' in cleaned_transactions_df
region_breakdown_corrected = cleaned_transactions_df.groupby('gateway_region_clean').agg(
    transaction_count=('transaction_id', 'count'),
    total_amount=('amount_usd', 'sum')
).reset_index()

# Rename columns for clarity
region_breakdown_corrected.rename(columns={'gateway_region_clean': 'Region', 'transaction_count': 'Transaction Count', 'total_amount': 'Total Amount'}, inplace=True)

print("Corrected Region Breakdown from Cleaned Transactions:")
display(region_breakdown_corrected)

Corrected Region Breakdown from Cleaned Transactions:


,Region,Transaction Count,Total Amount
0,APAC,13,50177.5
1,EU,4,18886.0
2,MISSING,9,32416.5
3,US,4,14600.0


In [53]:
region_breakdown_corrected.to_csv('region_breakdown_corrected.csv', index=False)
print("Exported 'region_breakdown_corrected.csv'")

Exported 'region_breakdown_corrected.csv'


### Merchant Performance Summary

In [54]:
# Group by merchant_name_clean and aggregate transaction count and total amount
merchant_performance_summary = cleaned_transactions_df.groupby('merchant_name_clean').agg(
    transaction_count=('transaction_id', 'count'),
    total_amount=('amount_usd', 'sum')
).reset_index()

# Rename columns for clarity
merchant_performance_summary.rename(columns={'merchant_name_clean': 'Merchant Name', 'transaction_count': 'Transaction Count', 'total_amount': 'Total Amount'}, inplace=True)

print("Merchant Performance Summary:")
display(merchant_performance_summary)

Merchant Performance Summary:


,Merchant Name,Transaction Count,Total Amount
0,Alpha Mart,11,40812.0
1,Beta Stores,11,41782.0
2,City Pharma,2,8640.0
3,Delta Travels,4,14600.0
4,Eco Home,2,10246.0


In [55]:
merchant_performance_summary.to_csv('merchant_performance_summary.csv', index=False)
print("Exported 'merchant_performance_summary.csv'")

Exported 'merchant_performance_summary.csv'
